# HWT — OPTIONAL fine-tune (Colab, GPU)

**Only if pretrained HWT underfits your style.** Try the official custom-
handwriting demo first: github.com/ankanbhunia/Handwriting-Transformers
(it has a ready-made `demo_custom_handwriting.ipynb` — often all you need).

- Repo: https://github.com/ankanbhunia/Handwriting-Transformers (ICCV 2021)
- Base: official iam_model.pth (never from scratch)
- Checkpoints: Drive-backed-up, auto-resumed

In [ ]:
!nvidia-smi

In [ ]:
MODEL_NAME = 'hwt'

# 2. Clone + official pretrained bundle (models + IAM pickle)
!git clone --depth 1 https://github.com/ankanbhunia/Handwriting-Transformers /content/HWT
%cd /content/HWT
!pip install -q --upgrade --no-cache-dir gdown
!gdown --id 16g9zgysQnWk7-353_tMig92KsZsrcM6k && unzip -o files.zip && rm files.zip

In [ ]:
# ============================================================
# CHECKPOINT RESUME — survives Colab tier switches
# ------------------------------------------------------------
# Every epoch, the trainer copies its latest checkpoint to YOUR
# Google Drive. If Colab disconnects / your free tier ends:
#   1. Open THIS notebook in a new Colab session (any tier).
#   2. Run all cells top-to-bottom again.
#   3. The trainer detects the Drive checkpoint and RESUMES
#      from the last epoch instead of starting over.
# Nothing is ever stored only on Colab's throwaway disk.
# ============================================================
from google.colab import drive
import shutil, pathlib

drive.mount('/content/drive')
DRIVE_DIR = pathlib.Path('/content/drive/MyDrive/textwritter_checkpoints') / MODEL_NAME
DRIVE_DIR.mkdir(parents=True, exist_ok=True)

LOCAL_CKPT = pathlib.Path('checkpoints')
LOCAL_CKPT.mkdir(exist_ok=True)

def resume_from_drive():
    """Copy newest Drive checkpoint back to local disk. Returns path or None."""
    ckpts = sorted(DRIVE_DIR.glob('*.pth'), key=lambda p: p.stat().st_mtime)
    if ckpts:
        dst = LOCAL_CKPT / ckpts[-1].name
        shutil.copy2(ckpts[-1], dst)
        print(f'[resume] found {ckpts[-1].name} on Drive -> resuming')
        return dst
    print('[resume] no Drive checkpoint -> starting from pretrained base')
    return None

def save_to_drive(src: pathlib.Path):
    shutil.copy2(src, DRIVE_DIR / src.name)
    print(f'[checkpoint] {src.name} backed up to Drive')


## 3. Your handwriting data

HWT custom training expects a pickle of `{writer: [{img, label}, ...]}`
(see repo INSTALL.md). Upload word-crops + labels as described there;
this cell converts a simple zip into that pickle.

In [ ]:
from google.colab import files
uploaded = files.upload()  # my_words.zip with images + labels.txt

import zipfile, pathlib, pickle
from PIL import Image
DATA = pathlib.Path('/content/data'); DATA.mkdir(exist_ok=True)
for name in uploaded:
    with zipfile.ZipFile(name) as z: z.extractall(DATA)

labels = {}
for line in (DATA / 'labels.txt').read_text().splitlines():
    fn, txt = line.split('\t')
    labels[fn.strip()] = txt.strip()

samples = [{'img': Image.open(DATA / fn).convert('RGB'), 'label': txt}
           for fn, txt in labels.items()]
bundle = {'train': [{'me': samples}], 'test': [{'me': samples[:5]}]}
with open('files/custom.pickle', 'wb') as f:
    pickle.dump(bundle, f)
print(f'{len(samples)} word samples packed -> files/custom.pickle')

## 4. Fine-tune from iam_model.pth (2–4 h on T4)

In [ ]:
resume_ckpt = resume_from_drive()
BASE = str(resume_ckpt or 'files/iam_model.pth')

!python train.py \
    --dataname custom \
    --modelname custom_model \
    --pretrained $BASE \
    --lr 1e-5 \
    --epochs 15 \
    2>&1 | tail -20

import pathlib
for ckpt in sorted(pathlib.Path('files').glob('custom_model*.pth')):
    save_to_drive(ckpt)

## 5. Use it

Download `custom_model*.pth` from Drive, place in HWT `files/`, point the
repo's inference at it. Colab cut you off? Run-All resumes from Drive.